# SCE-Net (modern) — train/val на готовых триплетах, test на парах

Ниже — **практический пайплайн под ваш формат данных**:

- `train_triplets.csv` и `val_triplets.csv` уже готовы и содержат триплеты;
- `test_pairs.csv` содержит пары с target (`good` / `bad`);
- условия (`condition`) как в UT-Zappos не используются.

Ноутбук сделан максимально подробно: каждая секция объясняет, что и зачем происходит.

## Формат файлов

### 1) Train/Val triplets
Ожидаемые колонки:
- `anchor_path` — путь к anchor изображению
- `positive_path` — путь к positive изображению
- `negative_path` — путь к negative изображению

### 2) Test pairs
Ожидаемые колонки:
- `sku1_path` — путь к первому товару
- `sku2_path` — путь ко второму товару
- `target` — `good` или `bad`

In [ ]:
# Если нужно, установите зависимости (раскомментируйте):
# !pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install -U transformers pandas scikit-learn pillow tqdm matplotlib

In [ ]:
# ============================
# 1. Импорты и воспроизводимость
# ============================

import os                        # работа с файловой системой
import random                    # генерация случайных чисел
from dataclasses import dataclass
from pathlib import Path         # удобная работа с путями
from typing import Dict

import numpy as np               # массивы/математика
import pandas as pd              # таблицы
from PIL import Image            # чтение картинок
from tqdm.auto import tqdm       # прогресс-бары

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoModel, AutoProcessor
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42                        # фиксируем seed для повторяемости экспериментов
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Выбираем GPU, если доступен; иначе CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# ==================================
# 2. Пути к данным и базовая проверка
# ==================================

TRAIN_TRIPLETS_CSV = 'data/train_triplets.csv'
VAL_TRIPLETS_CSV   = 'data/val_triplets.csv'
TEST_PAIRS_CSV     = 'data/test_pairs.csv'
IMG_ROOT = ''  # если в csv относительные пути, можно задать корневую папку

train_df = pd.read_csv(TRAIN_TRIPLETS_CSV)
val_df   = pd.read_csv(VAL_TRIPLETS_CSV)
test_df  = pd.read_csv(TEST_PAIRS_CSV)

# Проверяем обязательные колонки для triplets
triplet_cols = {'anchor_path', 'positive_path', 'negative_path'}
for name, df_ in [('train', train_df), ('val', val_df)]:
    miss = triplet_cols - set(df_.columns)
    if miss:
        raise ValueError(f'{name} csv missing columns: {miss}')

# Проверяем обязательные колонки для test pairs
pair_cols = {'sku1_path', 'sku2_path', 'target'}
miss = pair_cols - set(test_df.columns)
if miss:
    raise ValueError(f'test csv missing columns: {miss}')

# Нормализуем target в числовой вид: good->1, bad->0
test_df['label'] = (test_df['target'].str.lower() == 'good').astype(int)

# Если задан IMG_ROOT, добавляем префикс к путям
if IMG_ROOT:
    for col in ['anchor_path', 'positive_path', 'negative_path']:
        train_df[col] = train_df[col].apply(lambda p: str(Path(IMG_ROOT) / p))
        val_df[col]   = val_df[col].apply(lambda p: str(Path(IMG_ROOT) / p))
    for col in ['sku1_path', 'sku2_path']:
        test_df[col]  = test_df[col].apply(lambda p: str(Path(IMG_ROOT) / p))

print('train triplets:', len(train_df))
print('val triplets:', len(val_df))
print('test pairs:', len(test_df), '| positives:', int(test_df.label.sum()))

## 3) Датасеты

Ниже два Dataset-класса:
1. `TripletDataset` — для train/val;
2. `PairDataset` — для test.

Обратите внимание: мы НЕ используем `condition`-метки (в отличие от старого zappos-пайплайна).

In [ ]:
class TripletDataset(Dataset):
    """
    Датасет для готовых триплетов.
    Каждая строка csv -> (anchor, positive, negative).
    """
    def __init__(self, df: pd.DataFrame, processor):
        self.df = df.reset_index(drop=True)     # переиндексируем для безопасного __getitem__
        self.processor = processor               # image preprocessor от FashionCLIP

    def __len__(self):
        return len(self.df)                      # количество триплетов

    def _open_rgb(self, path: str) -> Image.Image:
        # Открываем изображение и приводим к RGB
        return Image.open(path).convert('RGB')

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]                  # берем строку по индексу

        a_img = self._open_rgb(row['anchor_path'])
        p_img = self._open_rgb(row['positive_path'])
        n_img = self._open_rgb(row['negative_path'])

        # processor возвращает dict, берем тензор pixel_values формы [1,3,H,W]
        a = self.processor(images=a_img, return_tensors='pt')['pixel_values'].squeeze(0)
        p = self.processor(images=p_img, return_tensors='pt')['pixel_values'].squeeze(0)
        n = self.processor(images=n_img, return_tensors='pt')['pixel_values'].squeeze(0)

        return {'anchor': a, 'positive': p, 'negative': n}


class PairDataset(Dataset):
    """
    Датасет для тестовых пар с метками совместимости.
    """
    def __init__(self, df: pd.DataFrame, processor):
        self.df = df.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def _open_rgb(self, path: str) -> Image.Image:
        return Image.open(path).convert('RGB')

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        i_img = self._open_rgb(row['sku1_path'])
        j_img = self._open_rgb(row['sku2_path'])

        i = self.processor(images=i_img, return_tensors='pt')['pixel_values'].squeeze(0)
        j = self.processor(images=j_img, return_tensors='pt')['pixel_values'].squeeze(0)

        return {'img_i': i, 'img_j': j, 'label': int(row['label'])}

## 4) Архитектура SCE-Net (без внешних condition-labels)

Состав:
- image encoder: `patrickjohncyh/fashion-clip`
- condition masks: `C in R^{M x D}`
- condition weight branch: `w = softmax(MLP([V_i, V_j]))`
- итоговый conditioned embedding: `E = w^T * [C_1⊙V, ..., C_M⊙V]`

In [ ]:
class ConditionWeightBranch(nn.Module):
    """
    По паре общих эмбеддингов (V_i, V_j) предсказывает веса условий w.
    """
    def __init__(self, emb_dim: int, num_conditions: int, hidden_dim: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim * 2, hidden_dim)    # concat(V_i, V_j) -> hidden
        self.act = nn.ReLU(inplace=True)                  # нелинейность
        self.drop = nn.Dropout(dropout)                   # регуляризация
        self.fc2 = nn.Linear(hidden_dim, num_conditions)  # hidden -> M logits

    def forward(self, v_i: torch.Tensor, v_j: torch.Tensor) -> torch.Tensor:
        x = torch.cat([v_i, v_j], dim=-1)                 # [B,2D]
        x = self.fc1(x)                                   # [B,H]
        x = self.act(x)
        x = self.drop(x)
        logits = self.fc2(x)                              # [B,M]
        w = F.softmax(logits, dim=-1)                     # [B,M], сумма по M = 1
        return w


class SCENet(nn.Module):
    def __init__(self, encoder_name='patrickjohncyh/fashion-clip', num_conditions=5, proj_dim=None, freeze_encoder=False):
        super().__init__()

        # 1) Загружаем pretrained FashionCLIP encoder
        self.encoder = AutoModel.from_pretrained(encoder_name)

        # 2) Определяем размер его image embeddings (D0)
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224)
            d0 = self.encoder.get_image_features(pixel_values=dummy).shape[-1]

        # 3) Если нужно, добавляем линейную проекцию D0 -> D
        self.emb_dim = proj_dim or d0
        self.proj = nn.Identity() if self.emb_dim == d0 else nn.Linear(d0, self.emb_dim)

        # 4) Learnable condition masks C: [M,D]
        self.num_conditions = num_conditions
        self.condition_masks = nn.Parameter(torch.empty(num_conditions, self.emb_dim))
        nn.init.xavier_uniform_(self.condition_masks)

        # 5) Ветка весов условий
        self.weight_branch = ConditionWeightBranch(self.emb_dim, num_conditions)

        # 6) Опционально замораживаем encoder
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

    def encode(self, pixel_values: torch.Tensor) -> torch.Tensor:
        # Получаем общий embedding V
        v = self.encoder.get_image_features(pixel_values=pixel_values)
        v = F.normalize(v, dim=-1)             # l2-нормировка стабилизирует обучение
        v = self.proj(v)
        return v

    def apply_conditions(self, v: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
        # v: [B,D], masks: [M,D], w: [B,M]

        # Строим masked embeddings O: [B,M,D]
        masked = v.unsqueeze(1) * self.condition_masks.unsqueeze(0)

        # Взвешиваем по w: E = w^T O -> [B,D]
        e = torch.bmm(w.unsqueeze(1), masked).squeeze(1)

        # Нормируем финальный embedding
        e = F.normalize(e, dim=-1)
        return e

    def forward_pair(self, img_i: torch.Tensor, img_j: torch.Tensor):
        # 1) Общие embeddings
        v_i = self.encode(img_i)
        v_j = self.encode(img_j)

        # 2) Веса условий по паре
        w = self.weight_branch(v_i, v_j)

        # 3) Conditioned embeddings
        e_i = self.apply_conditions(v_i, w)
        e_j = self.apply_conditions(v_j, w)

        return e_i, e_j, w

    def forward_triplet(self, anchor, positive, negative):
        # Пара (anchor, positive)
        e_a_pos, e_p, w_pos = self.forward_pair(anchor, positive)

        # Пара (anchor, negative)
        e_a_neg, e_n, w_neg = self.forward_pair(anchor, negative)

        # Усредняем две версии anchor embedding
        e_a = 0.5 * (e_a_pos + e_a_neg)

        return e_a, e_p, e_n, {'w_pos': w_pos, 'w_neg': w_neg}

## 5) Loss

Итоговая функция:
`L = L_triplet + λ1 * L1(C) + λ2 * L2(E)`

In [ ]:
@dataclass
class LossCfg:
    margin: float = 0.2
    lambda_l1: float = 1e-6
    lambda_l2: float = 1e-3


def compute_loss(e_a, e_p, e_n, masks, cfg: LossCfg):
    # Расстояние до positive
    d_pos = torch.norm(e_a - e_p, p=2, dim=-1)

    # Расстояние до negative
    d_neg = torch.norm(e_a - e_n, p=2, dim=-1)

    # Triplet hinge
    l_triplet = F.relu(d_pos - d_neg + cfg.margin).mean()

    # L1 по маскам
    l1_masks = masks.abs().mean()

    # L2 по эмбеддингам
    l2_embed = (e_a.pow(2).mean() + e_p.pow(2).mean() + e_n.pow(2).mean()) / 3.0

    # Финальный loss
    loss = l_triplet + cfg.lambda_l1 * l1_masks + cfg.lambda_l2 * l2_embed

    logs = {
        'loss': float(loss.item()),
        'triplet': float(l_triplet.item()),
        'l1_masks': float(l1_masks.item()),
        'l2_embed': float(l2_embed.item()),
        'd_pos': float(d_pos.mean().item()),
        'd_neg': float(d_neg.mean().item()),
    }
    return loss, logs

In [ ]:
# ============================================
# 6. Инициализация загрузчиков, модели, optimizer
# ============================================

ENCODER_NAME = 'patrickjohncyh/fashion-clip'
NUM_CONDITIONS = 5
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
WD = 1e-4

processor = AutoProcessor.from_pretrained(ENCODER_NAME)

train_ds = TripletDataset(train_df, processor)
val_ds   = TripletDataset(val_df, processor)
test_ds  = PairDataset(test_df, processor)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64,         shuffle=False, num_workers=4, pin_memory=True)

model = SCENet(encoder_name=ENCODER_NAME, num_conditions=NUM_CONDITIONS).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
loss_cfg = LossCfg()

print('train batches:', len(train_loader), '| val batches:', len(val_loader), '| test batches:', len(test_loader))

In [ ]:
def _to_device_triplet(batch):
    a = batch['anchor'].to(device, non_blocking=True)
    p = batch['positive'].to(device, non_blocking=True)
    n = batch['negative'].to(device, non_blocking=True)
    return a, p, n


@torch.no_grad()
def validate(model, loader, loss_cfg):
    model.eval()
    rows = []
    for batch in tqdm(loader, desc='val', leave=False):
        a, p, n = _to_device_triplet(batch)
        e_a, e_p, e_n, aux = model.forward_triplet(a, p, n)
        _, logs = compute_loss(e_a, e_p, e_n, model.condition_masks, loss_cfg)
        rows.append(logs)
    return pd.DataFrame(rows).mean().to_dict()


def train_epoch(model, loader, optimizer, loss_cfg):
    model.train()
    rows = []
    for batch in tqdm(loader, desc='train', leave=False):
        a, p, n = _to_device_triplet(batch)

        e_a, e_p, e_n, aux = model.forward_triplet(a, p, n)
        loss, logs = compute_loss(e_a, e_p, e_n, model.condition_masks, loss_cfg)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        rows.append(logs)
    return pd.DataFrame(rows).mean().to_dict()

In [ ]:
# =======================
# 7. Основной train loop
# =======================

os.makedirs('checkpoints', exist_ok=True)
best_val = float('inf')

for epoch in range(1, EPOCHS + 1):
    tr = train_epoch(model, train_loader, optimizer, loss_cfg)
    va = validate(model, val_loader, loss_cfg)

    print(f"Epoch {epoch:02d} | train_loss={tr['loss']:.4f} | val_loss={va['loss']:.4f} | "
          f"d_pos={va['d_pos']:.4f} | d_neg={va['d_neg']:.4f}")

    if va['loss'] < best_val:
        best_val = va['loss']
        torch.save({'model': model.state_dict(), 'epoch': epoch}, 'checkpoints/sce_best.pt')
        print('  -> best checkpoint saved')

## 8) Инференс на test-парах (без триплета)

На инференсе нужна только пара `(i, j)`:
1. `e_i, e_j, w = forward_pair(i, j)`
2. `score = -||e_i - e_j||`
3. считаем метрики AUC / AP.

In [ ]:
@torch.no_grad()
def evaluate_pairs(model, loader):
    model.eval()

    y_true = []
    y_score = []

    for batch in tqdm(loader, desc='test-pairs'):
        i = batch['img_i'].to(device, non_blocking=True)
        j = batch['img_j'].to(device, non_blocking=True)
        y = batch['label'].cpu().numpy()

        e_i, e_j, w = model.forward_pair(i, j)
        dist = torch.norm(e_i - e_j, p=2, dim=-1)
        score = (-dist).detach().cpu().numpy()

        y_true.extend(y.tolist())
        y_score.extend(score.tolist())

    auc = roc_auc_score(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    return {'AUC': auc, 'AP': ap}

In [ ]:
# Загружаем лучший checkpoint и считаем финальные метрики на test
ckpt = torch.load('checkpoints/sce_best.pt', map_location=device)
model.load_state_dict(ckpt['model'])

test_metrics = evaluate_pairs(model, test_loader)
print('TEST metrics:', test_metrics)

## 9) Краткий sanity-check по условиям (без condition labels)

Хотя condition labels нет, можно анализировать, какие маски чаще активируются:
- считаем `argmax(w)` на test-парах;
- строим распределение.

In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def inspect_condition_usage(model, loader):
    model.eval()
    winners = []
    for batch in tqdm(loader, desc='condition-usage'):
        i = batch['img_i'].to(device)
        j = batch['img_j'].to(device)
        _, _, w = model.forward_pair(i, j)
        winners.extend(torch.argmax(w, dim=-1).cpu().tolist())

    plt.figure(figsize=(8, 4))
    plt.hist(winners, bins=np.arange(model.num_conditions + 1) - 0.5, rwidth=0.85)
    plt.xticks(range(model.num_conditions))
    plt.xlabel('Condition index')
    plt.ylabel('Count')
    plt.title('How often each condition wins')
    plt.show()

inspect_condition_usage(model, test_loader)